# Pipeline explorer

Run one document through the pipeline and watch each stage. Change the settings
in the next cell, then **Run All**.

Every stage prints what it actually produced, followed by a note on what that
stage did and what tends to go wrong in it.


## Settings

Edit these four lines. Everything below follows from them.

| setting | options |
|---|---|
| `SAMPLE` | `1`–`10` for a document from the evaluation set, or `None` to use `PDF_PATH` |
| `PDF_PATH` | path to any PDF — used only when `SAMPLE = None` |
| `MODEL` | `'llama3.1:8b'`, `'gemma4:e4b'`, `'llama3.3:70b'` |
| `EXTRACTOR` | `'pdfplumber'`, `'docling'`, `'lighton'` |
| `RERUN` | `False` reuses saved output; `True` calls the model again |

> With `RERUN = False` this opens instantly if the combination has been run
> before. Set it to `True` after changing the prompt, or the notebook will show
> you output from the old one.


In [1]:
SAMPLE    = 1                  # 1-10, or None to use PDF_PATH
PDF_PATH  = 'data/input/pdfs/sample1.pdf'
MODEL     = 'llama3.1:8b'
EXTRACTOR = 'pdfplumber'
RERUN     = False


In [2]:
import json
import os
import time
from pathlib import Path

if Path.cwd().name == 'notebooks':
    os.chdir(Path.cwd().parent)

from dmpbridge.core import paths as P
from dmpbridge.core.pipeline import run_and_save
from dmpbridge.strategies.wholedoc import WholeDocStrategy

TAG = P.make_tag(MODEL, EXTRACTOR)

if SAMPLE is not None:
    pdf   = Path(f'data/input/pdfs/sample{SAMPLE}.pdf')
    stem  = f'sample{SAMPLE}'
    paths = {2: P.labeled_path(TAG, SAMPLE), 3: P.structured_path(TAG, SAMPLE),
             4: P.final_path(TAG, SAMPLE)}
else:
    # A PDF from outside the evaluation set: keep its output out of the
    # tagged folders, which are reserved for scored runs.
    pdf   = Path(PDF_PATH)
    stem  = pdf.stem
    scratch = P.OUTPUT_ROOT / 'explorer' / TAG
    paths = {n: scratch / f'{stem}__stage{n}.json' for n in (2, 3, 4)}

if not pdf.exists():
    raise FileNotFoundError(f'No PDF at {pdf}')

print(f'document   {pdf}')
print(f'model      {MODEL}')
print(f'extractor  {EXTRACTOR}')
print(f'tag        {TAG}')

missing = [n for n, p in paths.items() if not p.exists()]
if RERUN or missing:
    why = 'RERUN = True' if RERUN else f'stage(s) {missing} not yet produced'
    print(f'\nRunning the pipeline ({why}) — this calls {MODEL}.')
    t0 = time.time()
    strategy = WholeDocStrategy(model=MODEL, extractor=EXTRACTOR,
                                cache_dir=P.EXTRACTED_DIR / EXTRACTOR)
    run_and_save(strategy, pdf, paths[2], struct_path=paths[3], final_path=paths[4])
    print(f'done in {time.time() - t0:.1f} s')
else:
    print('\nUsing saved output. Set RERUN = True to call the model again.')


document   data\input\pdfs\sample1.pdf
model      llama3.1:8b
extractor  pdfplumber
tag        llama3.1-8b_pdfplumber_whole_doc

Using saved output. Set RERUN = True to call the model again.


## Stage 1 — read the PDF

The extractor turns pages into text blocks. Nothing is classified yet; this step
has no idea what a question or an answer is.


In [3]:
raw = json.loads((P.EXTRACTED_DIR / EXTRACTOR / f'{stem}.json').read_text(encoding='utf-8'))
blocks1 = raw if isinstance(raw, list) else raw.get('blocks', [])

print(f'{len(blocks1)} blocks\n')
for i, b in enumerate(blocks1[:12]):
    flags = ''.join(c for c, k in (('B', 'bold'), ('I', 'italic')) if b.get(k)) or '-'
    print(f"{i:>3}  p{b.get('page', '?')}  {flags:<3} {b.get('text', '')[:74]!r}")
if len(blocks1) > 12:
    print(f'... {len(blocks1) - 12} more')


44 blocks

  0  p1  -   'DATA MANAGEMENT AND SHARING PLAN'
  1  p1  -   'Element 1: Data Type:'
  2  p1  -   'A. Types and amount of scientific data expected to be generated in the pro'
  3  p1  -   'This secondary data analysis project will analyze deidentified data from 4'
  4  p1  -   'The studies include (i) the RISE Study, (ii) the SOL-VIDA Study, (iii) the'
  5  p1  -   'B. Scientific data that will be preserved and shared, and the rationale fo'
  6  p1  -   'As this is a secondary data analysis project, we will only be able to publ'
  7  p1  -   '(i) The RISE study'
  8  p1  -   '(ii) The iWATCH study'
  9  p1  -   '(iii) NHANES cohorts'
 10  p1  -   'Sufficient data from those datasets will be preserved to enable sharing to'
 11  p1  -   'about other studies.'
... 32 more


**What just happened.** The PDF became a flat list of text blocks, each carrying
position and font information (`B` = bold, `I` = italic). The model never sees the
PDF itself — only this list, so anything lost here is lost for good.

**What goes wrong here.** `pdfplumber` splits by *line*, so one paragraph arrives as
several blocks; a merging step rejoins wrapped lines before labeling. `docling` and
`lighton` segment by paragraph already. This stage is cached per extractor and
shared by every model, which is why changing `MODEL` above does not re-extract.


## Stage 2 — label every block

The whole document goes to the model in **one call**, and it returns a label and a
confidence for each block.


In [4]:
raw2 = json.loads(paths[2].read_text(encoding='utf-8'))
blocks2 = raw2 if isinstance(raw2, list) else raw2.get('blocks', [])

counts = {}
for b in blocks2:
    counts[b.get('label', '?')] = counts.get(b.get('label', '?'), 0) + 1
print('labels assigned:')
for lab, n in sorted(counts.items(), key=lambda x: -x[1]):
    print(f'  {lab:<22}{n:>4}')

print('\nfirst 12 blocks:')
for i, b in enumerate(blocks2[:12]):
    conf = b.get('confidence')
    c = f'{conf:.2f}' if isinstance(conf, (int, float)) else '  - '
    print(f"{i:>3}  {c}  [{b.get('label', '?'):<20}] {b.get('text', '')[:56]!r}")


labels assigned:
  answer.text             27
  section.title           14
  question.text            2
  title                    1

first 12 blocks:
  0  1.00  [title               ] 'DATA MANAGEMENT AND SHARING PLAN'
  1  1.00  [section.title       ] 'Element 1: Data Type:'
  2  1.00  [question.text       ] 'A. Types and amount of scientific data expected to be ge'
  3  1.00  [answer.text         ] 'This secondary data analysis project will analyze deiden'
  4  1.00  [answer.text         ] 'The studies include (i) the RISE Study, (ii) the SOL-VID'
  5  1.00  [section.title       ] 'B. Scientific data that will be preserved and shared, an'
  6  1.00  [answer.text         ] 'As this is a secondary data analysis project, we will on'
  7  1.00  [answer.text         ] '(i) The RISE study'
  8  1.00  [answer.text         ] '(ii) The iWATCH study'
  9  1.00  [answer.text         ] '(iii) NHANES cohorts'
 10  1.00  [answer.text         ] 'Sufficient data from those datasets will be preserve

**What just happened.** Each block now has one of five labels — `title`,
`section.title`, `section.description`, `question.text`, `answer.text` — plus the
model's own confidence. The label definitions come from
[`dmpbridge/prompts/system.py`](../dmpbridge/prompts/system.py).

**What goes wrong here.** Almost every pipeline error starts in this stage, and the
hardest boundary is `question.text` versus `section.title`: a lettered sub-item
(`A.`, `B.`, `C.`) under a heading is a *question*, but it looks like a heading.
Compare the count above against what you would expect — a document with 9 real
questions labelled with only 2 has already lost most of them here.


## Stage 3 — build the structure

The flat list becomes a nested document: sections, each holding questions, each
holding an answer. No model call — this is deterministic, driven entirely by the
labels from stage 2.


In [5]:
def unpack(path):
    """(title, sections) from the DMP-tool narrative schema."""
    tpl = json.loads(path.read_text(encoding='utf-8')).get('narrative', {}).get('template', {})
    return tpl.get('title', ''), tpl.get('section', [])


title3, sections3 = unpack(paths[3])
nq3 = sum(len(s.get('question', [])) for s in sections3)
print(f'title      {title3!r}')
print(f'sections   {len(sections3)}')
print(f'questions  {nq3}\n')

for s in sections3[:4]:
    print(f"SECTION  {s.get('title', '')[:66]!r}")
    if s.get('description'):
        print(f"         description: {s['description'][:60]!r}")
    for q in s.get('question', [])[:2]:
        ans = (q.get('answer') or {}).get('json', {}).get('answer', '')
        print(f"    Q  {str(q.get('text', ''))[:64]!r}")
        print(f"    A  {str(ans)[:64]!r}")
    print()
if len(sections3) > 4:
    print(f'... {len(sections3) - 4} more sections')


title      'DATA MANAGEMENT AND SHARING PLAN'
sections   14
questions  12

SECTION  'Element 1: Data Type:'
    Q  'A. Types and amount of scientific data expected to be generated '
    A  'This secondary data analysis project will analyze deidentified d'

SECTION  'B. Scientific data that will be preserved and shared, and the rati'
    Q  ''
    A  'As this is a secondary data analysis project, we will only be ab'

SECTION  'C. Metadata, other relevant data, and associated documentation:'
    Q  ''
    A  'In addition to the data described above, code and models will be'

SECTION  'Element 2: Related Tools, Software and/or Code:'
    Q  ''
    A  'Data will be analyzed with custom code by our statistical and co'

... 10 more sections


**What just happened.** Consecutive blocks sharing a label were merged, and the
hierarchy was rebuilt: every `section.title` opens a new section, every
`question.text` opens a question inside it, and following `answer.text` becomes that
question's answer. This is the shape the DMP Tool expects.

**What goes wrong here.** Nothing is invented at this stage, but stage 2's mistakes
change shape. A question mislabelled as a heading does not just lose one label — it
**opens a whole new section**, so one wrong label can restructure the document. That
is why the section count above may be far higher than the document really has.


## Stage 4 — apply the annotation rules

A deterministic pass from `data/input/Rules.xlsx`: where a question has no text, it
is filled from the section heading, then the section description, then the document
title — whichever exists first.


In [6]:
title4, sections4 = unpack(paths[4])
nq4 = sum(len(s.get('question', [])) for s in sections4)

print(f'sections   {len(sections3)} -> {len(sections4)}')
print(f'questions  {nq3} -> {nq4}')

before = [str(q.get('text', '')) for s in sections3 for q in s.get('question', [])]
after  = [str(q.get('text', '')) for s in sections4 for q in s.get('question', [])]
changed = [(b, a) for b, a in zip(before, after) if b != a]

print(f'\n{len(changed)} question(s) filled in by the rules')
for b, a in changed[:6]:
    print(f'   was {b[:56]!r}')
    print(f'   now {a[:56]!r}')
if not changed:
    print('   (nothing changed — every question already had text)')


sections   14 -> 14
questions  12 -> 12

10 question(s) filled in by the rules
   was ''
   now 'B. Scientific data that will be preserved and shared, an'
   was ''
   now 'C. Metadata, other relevant data, and associated documen'
   was ''
   now 'Element 2: Related Tools, Software and/or Code:'
   was ''
   now 'A. Repository where scientific data and metadata will be'
   was ''
   now 'B. How scientific data will be findable and identifiable'
   was ''
   now 'C. When and how long the scientific data will be made av'


**What just happened.** Stage 3 is kept unconverted and stage 4 is written beside
it, so the two can be compared — which is what the section above does.

**What goes wrong here.** The rules only fill questions that are *empty*. They
cannot repair a question that was mislabelled as a heading in stage 2, because such
a question is not blank — it is missing entirely. If nothing changed above, that is
usually why.


## Scoring — only when a reference annotation exists

For the 10 evaluation samples the output can be compared against a manual
annotation. For any other PDF there is nothing to compare against, and this section
reports that instead.


In [7]:
if SAMPLE is None:
    print('No reference annotation for a user-supplied PDF — nothing to score.')
    print('The output above is the pipeline result; correctness is for you to judge.')
else:
    from dmpbridge.evaluation.evaluate import (
        _confusion_from_match, _match_structured, extract_gold, micro_prf1,
        resolve_old_gt_path,
    )

    gold = extract_gold(resolve_old_gt_path(SAMPLE))
    records, no_gold = _match_structured(paths[3], gold)
    m = micro_prf1(_confusion_from_match(records, no_gold))

    print(f'sample{SAMPLE} scored on its own (Path A)\n')
    print(f"  TP {m['tp']:>3}   correct")
    print(f"  FP {m['fp']:>3}   wrong label, or produced with no counterpart")
    print(f"  FN {m['fn']:>3}   in the annotation, not produced correctly\n")
    print(f"  precision {m['precision']:.3f}")
    print(f"  recall    {m['recall']:.3f}")
    print(f"  f1-score  {m['f1']:.3f}")

    wrong = [r for r in records if r['pred_label'] and r['pred_label'] != r['gold_label']]
    if wrong:
        print(f'\nEvery wrong label ({len(wrong)}):')
        for r in wrong:
            print(f"   {r['gold_label']} -> {r['pred_label']}")
            print(f"       {r['pred_text'][:66]!r}")
    if no_gold:
        print(f'\nProduced but not in the annotation ({len(no_gold)}):')
        for text, label in no_gold[:6]:
            print(f'   [{label}] {text[:60]!r}')


sample1 scored on its own (Path A)

  TP  19   correct
  FP  10   wrong label, or produced with no counterpart
  FN   9   in the annotation, not produced correctly

  precision 0.655
  recall    0.679
  f1-score  0.667

Every wrong label (9):
   question.text -> section.title
       'B. Scientific data that will be preserved and shared, and the rati'
   question.text -> section.title
       'C. Metadata, other relevant data, and associated documentation:'
   answer.text -> question.text
       'The following data will be created as a result of this project:'
   question.text -> section.title
       'A. Repository where scientific data and metadata will be archived:'
   question.text -> section.title
       'B. How scientific data will be findable and identifiable:'
   question.text -> section.title
       'C. When and how long the scientific data will be made available:'
   question.text -> section.title
       'A. Factors affecting subsequent access, distribution, or reuse of '
   que

**Reading this.** A block is matched to the annotation by shared words, then judged
on its label. `f1-score` combines precision and recall and stays low unless both are
high, so a model cannot score well by labelling very little or by labelling
everything.

The list of wrong labels is usually more useful than the score. If one mistake
repeats — the same `question.text -> section.title` over and over — that is a prompt
or capability problem worth fixing, not ten separate accidents.

---

**Try next:** change `MODEL` in the settings cell and Run All. Extraction is cached,
so only the labeling re-runs, and the differences you see are the model's alone.
